In [3]:
import tensorflow as tf
import os
import glob
import numpy as np
from model import build_1d_cnn_model
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES, fast_gpu_map
cnn_model=build_1d_cnn_model(None,INPUT_SHAPE,OUTPUT_DIM_NOTES,False)
cnn_model.summary()
tf.keras.utils.plot_model(cnn_model,to_file='cnn_model.png',show_shapes=True)
# cnn_model.load_weights('/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/checkpoints/guitarmidi_epoch02_valAcc0.9551.weights.h5')#
cnn_model.load_weights('guitarmidi.weights.h5')

input_data_dir = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices'
input_filepaths = sorted(glob.glob(os.path.join(input_data_dir, '**', 'input', 'data.tfrecord'), recursive=True))
# train_dataset = tf.data.Dataset.from_tensor_slices((input_filepaths))
# train_dataset=train_dataset.shuffle(buffer_size=len(input_filepaths))
# train_dataset=train_dataset.take(100)
# train_dataset = train_dataset.map(tf_load_sample_from_files, num_parallel_calls=tf.data.AUTOTUNE)
def representative_data_gen():
    # Use TFRecordDataset to actually read the files
    # We only need a few samples to calibrate quantization
    raw_dataset = tf.data.TFRecordDataset(input_filepaths).take(100)
    
    # Map using your existing loading function
    calib_dataset = raw_dataset.map(lambda path: fast_gpu_map(path, 0, training=False))
    
    for input_value, _ in calib_dataset.batch(1):
        # input_value is the (312, 256, 1) tensor
        yield [input_value]

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)

converter.optimizations = [ tf.lite.Optimize.DEFAULT]
# converter.representative_dataset = representative_data_gen
# converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
# converter.target_spec.supported_types = [tf.int8] 

# converter.target_spec.supported_types = [tf.float16]
# converter.target_spec._experimental_supported_accumulation_type = tf.dtypes.float16
# Perform the conversion
tflite_model = converter.convert()


# converter=tf.lite.TFLiteConverter.from_keras_model(cnn_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model=converter.convert()

with open('guitarmidi.tflite','wb') as f:
    f.write(tflite_model)
print("TFLite model saved as guitarmidi.tflite")

Initial input shape: (None, 312, 512)
After first Conv2D: (None, 156, 64)
Extracting string 1 from filters 0 to 26
String 1 section shape: (None, 26, 64)
String 1 after first Conv1D: (None, 26, 128)
Extracting string 2 from filters 26 to 52
String 2 section shape: (None, 26, 64)
String 2 after first Conv1D: (None, 26, 128)
Extracting string 3 from filters 52 to 78
String 3 section shape: (None, 26, 64)
String 3 after first Conv1D: (None, 26, 128)
Extracting string 4 from filters 78 to 104
String 4 section shape: (None, 26, 64)
String 4 after first Conv1D: (None, 26, 128)
Extracting string 5 from filters 104 to 130
String 5 section shape: (None, 26, 64)
String 5 after first Conv1D: (None, 26, 128)
Extracting string 6 from filters 130 to 156
String 6 section shape: (None, 26, 64)
String 6 after first Conv1D: (None, 26, 128)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 312, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_2 (Reshape) │ (None, 312, 256,  │          0 │ input_layer_1[0]… │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 312, 32,   │        272 │ reshape_2[0][0]   │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_9       │ (None, 312, 32,   │          0 │ conv2d_1[0][0]    │
│ (LeakyReLU)         │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_3 (Reshape) │ (None, 312, 512)  │          0 │ leaky_re_lu_9[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, 312, 32)   │     81,952 │ reshape_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 312, 32)   │        128 │ conv1d_8[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_10      │ (None, 312, 32)   │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_8 │ (None, 312, 32)   │          0 │ leaky_re_lu_10[0… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_9 (Conv1D)   │ (None, 312, 64)   │     10,304 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 312, 64)   │        256 │ conv1d_9[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_11      │ (None, 312, 64)   │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_7     │ (None, 156, 64)   │          0 │ leaky_re_lu_11[0… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_9 │ (None, 156, 64)   │          0 │ max_pooling1d_7[… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_6 (Lambda)   │ (None, 26, 64)    │          0 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_7 (Lambda)   │ (None, 26, 64)    │          0 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_8 (Lambda)   │ (None, 26, 64)    │          0 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_9 (Lambda)   │ (None, 26, 64)    │          0 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_10 (Lambda)  │ (None, 26, 64)    │          0 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 441,713 (1.69 MB)

 Trainable params: 439,985 (1.68 MB)

 Non-trainable params: 1,728 (6.75 KB)

INFO:tensorflow:Assets written to: /tmp/tmp5jhb1o44/assets


INFO:tensorflow:Assets written to: /tmp/tmp5jhb1o44/assets


Saved artifact at '/tmp/tmp5jhb1o44'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 312, 256, 1), dtype=tf.float32, name='keras_tensor_71')
Output Type:
  TensorSpec(shape=(None, 129), dtype=tf.float32, name=None)
Captures:
  123591449065040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123591449066192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123591449065808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123591449066768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123591449065424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123591449065616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123591449065232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123591449066384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123591449067344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123591449066576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123591447

W0000 00:00:1770758792.855438  911745 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1770758792.855453  911745 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1770758792.855647  911745 reader.cc:83] Reading SavedModel from: /tmp/tmp5jhb1o44
I0000 00:00:1770758792.857028  911745 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1770758792.857033  911745 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp5jhb1o44
I0000 00:00:1770758792.868558  911745 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1770758792.872214  911745 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1770758792.941649  911745 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmp5jhb1o44
I0000 00:00:1770758792.957884  911745 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 102246 microseconds.
I0000 00:00:1770758792.979787  91174